In [ ]:
# imports
import MDAnalysis as mda
import pandas as pd
import nglview as nv
from pathlib import Path
from tqdm.notebook import tqdm

# local imports
from setup_structures import assign_segment_ids, identify_subaggregates
from merge_structures import merge_structures
from validation import check_cluster_overlaps, order_structure, calculate_universe_epot

# A workflow to merge CALVADOS protein structures

### Define protein names and pdb file paths

In [ ]:
proteins = ["hpl2", "lin13", "met2", "lin65"]
T = 260.15
protein_file_names = {prot: f"inputs/single_comp_structures/{prot}_config_{T}K.pdb" for prot in proteins}
protein_dimensions = {prot: [600,600,3000,90,90,90] for prot in proteins}

### Preparation: setup structures

➔ load structures<br>
➔ identify proteins & clusters<br>
➔ calculate cluster cylinders

In [ ]:
# load the structure files, assign unique segment IDs for every protein, identify aggregates in the structures
# and save relevant paramenters in a pandas dataframe
existing_segids = {}
protein_data = {}
for prot in tqdm(protein_file_names, desc="Loading structure files", leave=True, unit="Files"):
    x = mda.Universe(protein_file_names[prot])
    u = x.copy()
    tqdm.write(f"Created universe for {prot}")

    # Add unique segment IDs to differentiate between protein chains
    u, prefix = assign_segment_ids(u, prot, existing_segids)
    tqdm.write(f"Assigned segment IDs to {prot}")

    # assign known universe dimensions
    u.dimensions = protein_dimensions[prot]

    # Analyze chains and clusters
    clusters = identify_subaggregates(u)
    tqdm.write(f"Identified clusters in {prot}")
    protein_data[prot] = {
        'segment_prefix': prefix,
        'universe': u,
        "n_atoms": len(u.atoms),
        'subaggregates': clusters,
        'rg_min_cluster': min([cluster["cluster"].radius_of_gyration() for cluster in clusters]),
        'rg_max': max([prot.atoms.radius_of_gyration() for prot in u.segments]),
        "n_proteins": len(u.segments), 
        "box_dimensions": u.dimensions
    }

proteins_df = pd.DataFrame.from_dict(protein_data, orient='index')
proteins_df

### Main loop: merge structures

➔ calculate box dimensions<br>
➔ place the structure with least clusters<br>
➔ evaluate z-offsets for other structures and sort a placement queue accordingly<br>
➔ place strucutures one at a time, save orphans to a list<br>
➔ find placement for orphans by scanning through z, y, z offsets

In [ ]:
merged = merge_structures(proteins_df)

In [ ]:
view = nv.show_mdanalysis(merged.atoms)
view.add_unitcell()
view

In [ ]:
merged.atoms.write("outputs/merged_structure.pdb")

### Validation

➔ check if any cluster overlaps with another

In [ ]:
cluster_overlaps = check_cluster_overlaps(merged, proteins_df, cutoff=10.0)
print(f"--- Final Report: {cluster_overlaps} inter-cluster collisions found ---")

In [ ]:
ordered_merged = order_structure(merged, ["A", "B", "C", "D"])
output_path = Path("outputs/merged_structure_ordered.pdb")
ordered_merged.atoms.write(output_path)
merged_epot = calculate_universe_epot(
    system_xml=Path("inputs/all_comps_condensed_260.15K.xml"),
    universe=ordered_merged,
    pdb_path=output_path, 
    T=T
)
print(f"Potential Energy: {merged_epot:.2f} kJ/mol")

In [ ]:
orig_structures = [mda.Universe(file_name).atoms for file_name in protein_file_names.values()]
arbitrary_structure = mda.Merge(*orig_structures)
arbitrary_structure.dimensions = [600, 600, 3000, 90, 90, 90]
arbitrary_structure.atoms.wrap(compound="atoms")
view = nv.show_mdanalysis(arbitrary_structure.atoms)
view.add_unitcell()
view

In [ ]:
arbitrary_path = Path("outputs/arbitrary.pdb")
arbitrary_structure.atoms.write(output_path)
arbitrary_epot = calculate_universe_epot(
    system_xml=Path("inputs/all_comps_condensed_260.15K.xml"),
    universe=arbitrary_structure,
    pdb_path=arbitrary_path
)

print(f"Potential Energy (rational placement): {merged_epot:6e} kJ/mol")
print(f"Potential Energy (arbitrary placement): {arbitrary_epot:6e} kJ/mol")